# 节点 4：规则建议与可选 LLM 建议

这一节点把单件物品状态整理成今天可以直接执行的任务计划；模型只负责把真实事实说得更自然。

## 1. 本节点目标

先用确定性规则找出到期清洗、适合晾晒和优先级，再选择是否交给 LLM 整理。没有密钥、没有模型名或调用失败时，同一份规则计划仍然可用。

## 2. 完成结果与验收

- “需要关注”只列到清洗周期且今天尚未清洗的物品。
- 今日计划按高、中、低优先级排序并解释原因。
- 无密钥时使用规则建议；fake client 可验证 AI 路径。
- 模型超时或空输出时自动降级。
- 发送给模型的是精简 JSON，不包含密钥和图片。
- 页面加入可选图片上传和紧凑天气框；建议能力随后统一合并到智能助手。
- 全量 42 项测试通过。

## 3. 本节点文件结构

```text
src/smart_laundry/recommendations.py  规则计划、上下文和 LLM 降级
src/smart_laundry/image_storage.py    图片验证、压缩与随机文件名
app.py                               紧凑首页、图片上传和建议页
tests/test_recommendations.py         规则、fake client 和降级测试
tests/test_image_storage.py           图片格式、尺寸和错误测试
notebooks/05_llm_recommendations.ipynb
```

## 4. 关键代码解释

`build_rule_plan()` 读取已经计算好的物品事实，根据清洗到期比例和天气生成 `PlanTask`，再按优先级排序。这一步完全不调用模型。

`generate_advice()` 先生成规则计划。缺少 `OPENAI_API_KEY` 或 `OPENAI_MODEL` 时直接返回规则文本；配置完整时调用 Responses API，并设置 `store=False`、输出长度上限和明确 instructions。SDK 支持的 `output_text` 用来读取最终文本，避免假设输出数组第一项一定是消息。

模型调用处于第三方边界，任何异常都会映射成安全的规则建议，不会让页面崩溃。

In [ ]:
def priority_from_ratio(ratio):
    if ratio is None or ratio >= 1.5:
        return '高'
    if ratio >= 1.0:
        return '中'
    return '低'

print(priority_from_ratio(1.8), priority_from_ratio(1.1), priority_from_ratio(0.8))

## 5. 数据流

```mermaid
flowchart LR
 A[物品与天气事实] --> B[规则计划]
 B --> C{密钥和模型已配置?}
 C -->|否| D[规则建议]
 C -->|是| E[Responses API]
 E -->|成功| F[AI 建议]
 E -->|超时/异常/空输出| D
 D --> G[页面展示]
 F --> G
```

## 6. 关键概念

- **规则引擎**：用明确条件生成稳定结果。
- **优先级**：帮助用户先做最紧急的任务。
- **结构化上下文**：用精简 JSON 提供模型需要的事实。
- **provider abstraction**：业务层不散落模型 SDK 调用。
- **fake client**：测试中模拟模型成功或失败。
- **fallback**：外部模型失败后仍能交付规则结果。
- **max output tokens**：限制模型输出长度和成本。

## 7. 为什么这样设计

模型不能决定数据库事实和是否到期，只能重新组织规则结果。这样建议具备自然语言表达，同时保留可解释性和无密钥演示能力。当前要求用户显式配置模型名，不擅自选择可能变化的默认模型。图片也不会发送给模型；上传功能目前只用于本地区分物品。

## 8. 常见错误与排查

1. **一直显示规则建议**：同时检查 `OPENAI_API_KEY` 和 `OPENAI_MODEL`。
2. **模型超时**：页面会降级，先确认规则计划正常，再检查网络和额度。
3. **兼容服务不支持 Responses API**：更换支持该接口的服务，或后续增加适配器。
4. **建议编造内容**：减少上下文字段、强化 instructions，并始终展示规则事实。
5. **图片无法上传**：检查是否为 JPG、PNG、WebP 且不超过 5 MB。
6. **相似被褥认错**：为物品上传实拍图并写清备注。

## 9. 面试可能追问

**问：为什么先规则再 LLM？** 答：事实和优先级需要稳定，模型只增强表达；无密钥和故障时仍可用。

**问：如何测试模型路径？** 答：注入 fake client，分别返回固定 output_text 或抛出超时。

**问：如何防止泄露密钥？** 答：只从环境变量读取，既不进入上下文，也不写日志和数据库。

**问：为什么图片暂不自动动漫化？** 答：先完成识别物品的上传闭环，再单独评估图像生成成本、隐私和确认流程。

## 10. 必须掌握的最少知识

需要理解规则计划永远先生成；模型只是可选表达层；环境变量决定是否启用 AI；任何模型失败都返回规则结果；图片当前只存本机，不进入模型。

## 11. 可自测小题

1. 为什么到期判断不能交给 LLM？
2. 缺少模型密钥时页面会怎样？
3. fake client 解决了什么测试问题？
4. 为什么模型输入使用 JSON？
5. 上传图片现在会不会发给模型？

<details><summary>参考答案</summary>

1. 日期事实应稳定可测。2. 使用规则建议。3. 不依赖真实网络和费用。4. 字段清晰且易控制。5. 不会，只保存在本地。

</details>

## 12. 动手小练习

1. 修改优先级示例中的比例，观察结果。
2. 不配置密钥点击“生成今日洗晒计划”，确认仍有计划。
3. 为两件相似物品分别上传照片，观察卡片缩略图。

## 13. 本节点术语表

| 术语 | 简单解释 |
|---|---|
| rule plan | 规则生成的可执行计划 |
| LLM provider | 提供模型调用的服务 |
| Responses API | 生成文本、JSON或工具调用响应的接口 |
| output_text | SDK 汇总后的文本输出 |
| fallback reason | 为什么切回规则模式 |
| local upload | 只保存在本机的上传文件 |
| thumbnail | 用于快速识别的小尺寸图片 |

## 14. 下一节点连接

下一节点会把天气查询和物品查询注册为正式工具，让 Agent 根据自然语言请求主动选择工具，而不是由页面提前把所有事实都准备好。